In [1]:
"""
ETA to scrape everything will roughly take 1hr.
"""

#%pip install bs4 requests fake_useragent lxml pandas numpy urllib3

'\nETA to scrape everything will roughly take 1hr.\n'

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# DEPENDENCIES

In [3]:
from bs4 import BeautifulSoup
import requests
from fake_useragent import UserAgent
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
import pandas as pd
import numpy as np
import threading 
from concurrent.futures import ThreadPoolExecutor
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [4]:
lock = threading.Lock()

fight_details = []
new_fight_links_all = []
winner_names = []
fighter_detail_data = []

In [5]:
MAX_THREADS = 3 # change this to adjust the number of concurrent threads

ua = UserAgent()
chrome = ua.chrome

HEADER = {
    'User-Agent' : chrome
}

In [6]:
def create_session(): # Create a session with retry strategy
    """Create a requests session with retry strategy for handling network issues."""
    # This function sets up a session with a retry strategy to handle network issues. 
    
    session = requests.Session()
    retry_strat = Retry(
        backoff_factor=7, # Wait time between retries increases exponentially
        total=10, # Total number of retries
        status_forcelist= [429, 500, 502, 503, 504], # Retry on these status codes
        allowed_methods=['GET']
    )
    adapter = HTTPAdapter(max_retries= retry_strat)
    session.mount('https://', adapter)
    session.mount('http://', adapter)
    return session

session = create_session()

# Scrapping the event links

In [7]:
ufc_link = "http://ufcstats.com/statistics/events/completed?page=all"

respone = session.get(ufc_link)

text = respone.text
soup = BeautifulSoup(text, 'lxml')

event_links_soup = soup.find_all('a', class_ = 'b-link b-link_style_black')

event_links = [link['href'] for link in event_links_soup] # Extracting href attributes from the links

print(len(event_links), "events found")

772 events found


# Scrapping the event info

In [8]:
def get_event_data(item): # Function to scrape event data
    """Scrape event data from the given link."""
    idx, link = item
    link = link.strip()
    response = session.get(link, headers=HEADER, timeout= 15)
    response.raise_for_status()
    if (response.status_code == 200):
        soup = BeautifulSoup(response.text, 'lxml')
                
        event_id = link[-16:]
        date_loc_list = soup.find_all('li', 'b-list__box-list-item')
        date = date_loc_list[0].text.replace("Date:", "").strip()
        location = date_loc_list[1].text.replace("Location:", "").strip()
        fight_links = soup.find_all('tr', class_ = 'b-fight-details__table-row b-fight-details__table-row__hover js-fight-details-click')
        for i in fight_links:
            winner_name = None
            winner_id = None
            w_l_d = i.find('i', class_ = "b-flag__text").text
            fight_id = i['data-link'][-16:]
            # print(w_l_d)
            if w_l_d == "win":
                players = i.find('td', class_ = "b-fight-details__table-col l-page_align_left")
                players = players.find_all('a', class_= "b-link b-link_style_black")
                winner_name = players[0].text.strip()
                winner_id = players[0]['href'][-16:]
            # Making the data
            data_dic = {
                "event_id" : event_id,
                "fight_id" : fight_id,
                "date" : date,
                "location" : location,
                "winner" : winner_name,
                "winner_id" : winner_id
            }
            new_fight_links_all.append(i['data-link'])
            winner_names.append(data_dic)
        # print(f"Scrapped : {link}, {idx+1} / {len(event_links)}")
        idx += 1
    else:
        print("Could'nt retrive the link." + str(response.status_code))

with ThreadPoolExecutor(max_workers= MAX_THREADS) as executor:
    results = [executor.submit(get_event_data, item) for item in enumerate(event_links)]
    for r in results:
        r.result()

df_winner = pd.DataFrame(data=winner_names)
df_winner.to_csv("event_details.csv", index = False)
print(f"Successfully scrapped {len(df_winner)} event data.")
df_winner

Successfully scrapped 8675 event data.


,event_id,fight_id,date,location,winner,winner_id
0,872b018076f831b0,eb6d9fef6c374830,"May 02, 2026","Perth, Western Australia, Australia",Carlos Prates,7ee0fd831c0fe7c3
1,872b018076f831b0,5c7c38dece435cde,"May 02, 2026","Perth, Western Australia, Australia",Quillan Salkilld,17734443a833cdf7
2,872b018076f831b0,afec383a96893ec5,"May 02, 2026","Perth, Western Australia, Australia",Steve Erceg,32ab52e5de93092d
3,872b018076f831b0,cfc884a249141cba,"May 02, 2026","Perth, Western Australia, Australia",Marwan Rahiki,6eedb757f13b9978
4,872b018076f831b0,a564219e7a964aa6,"May 02, 2026","Perth, Western Australia, Australia",Brando Pericic,d0fd0d9ee560dae7
...,...,...,...,...,...,...
8670,1a49e0670dfaca31,5167a5aec41f8b6d,"September 09, 1994","Charlotte, North Carolina, USA",Ken Shamrock,63b65af1c5cb02cb
8671,1a49e0670dfaca31,f65b000b7e450bcf,"September 09, 1994","Charlotte, North Carolina, USA",Royce Gracie,429e7d3725852ce9
8672,1a49e0670dfaca31,ca837b002a318766,"September 09, 1994","Charlotte, North Carolina, USA",Harold Howard,a2b06ca02bca14c0
8673,1a49e0670dfaca31,56315d5ac11594a5,"September 09, 1994","Charlotte, North Carolina, USA",Ken Shamrock,63b65af1c5cb02cb


# Scraping the fight info

In [9]:
def get_fight_data(item): # Function to scrape fight data
    """Scrape fight data from the given link."""
    idx, link = item
    link = link.strip()
    try:
        response = session.get(link, headers=HEADER, timeout=15)
        response.raise_for_status() 
        
        soup = BeautifulSoup(response.text, 'lxml')
        
        # event name
        event_name = soup.find('a', class_ = "b-link").text.strip()
        # event id
        event_id = soup.find('a', class_ = "b-link")['href'][-16:]
        # fight id
        fight_id = link[-16:]
        
        # fighter names
        fighter_nams = soup.find_all('a', class_ = 'b-link b-fight-details__person-link')
        r_name = fighter_nams[0].text.strip()
        b_name = fighter_nams[1].text.strip()
        
        # fighter ids
        r_id = fighter_nams[0]['href'].strip()[-16:]
        b_id = fighter_nams[1]['href'].strip()[-16:]
        
        # title fight & division
        division_info = soup.find('i', class_= 'b-fight-details__fight-title').text.lower()
        is_title_fight = 0
        if 'title' in division_info:
            is_title_fight = 1
        division_info = division_info.replace('ufc', "")
        division_info = division_info.replace("title", "")
        division_info = division_info.replace("bout", "").strip()
        
        # method
        method = soup.find('i', style = 'font-style: normal').text.strip()
        
        
        p_tag_with_fight_detail = soup.find('p', class_ = "b-fight-details__text")
        fight_details_list = p_tag_with_fight_detail.find_all('i', class_ = 'b-fight-details__text-item')
        # finish-round
        finish_round = int(fight_details_list[0].text.lower().replace("round:", "").strip())
        # match-time
        match_timestamp = fight_details_list[1].text.lower().replace("time:", "").strip()
        match_timestamp_splited = match_timestamp.split(":")
        match_time_sec = int(match_timestamp_splited[0]) * 60 + int(match_timestamp_splited[-1])
        # total-round
        total_rounds = fight_details_list[2].text.lower().replace("time format:", "").strip()
        if total_rounds == "No Time Limit".lower():
            total_rounds = None
        else :
            total_rounds = int(total_rounds[0])
        # referee
        referee = fight_details_list[3].text.replace("Referee:", "").strip()
        
        
        # totals, SIG. STR.
        tables = soup.find_all('table', style = "width: 745px")
        
        # TOTALS TABLE
        if len(tables) > 0:
            table1 = tables[0]
            td_1_list = table1.find_all('td', class_ = 'b-fight-details__table-col')
            # KD
            kd_players = td_1_list[1].text.split()
            r_kd, b_kd = int(kd_players[0]), int(kd_players[1])
            # sig. str.
            sig_str_players = td_1_list[2].text.split() 
            r_sig_str_landed = int(sig_str_players[0])
            r_sig_str_atmpted = int(sig_str_players[2])
            b_sig_str_landed = int(sig_str_players[3])
            b_sig_str_atmpted = int(sig_str_players[5])
            # sig_str_acc
            sig_str_acc = td_1_list[3].text.split() 
            r_sig_str_acc = int(sig_str_acc[0].replace("%", "")) if sig_str_acc[0] != "---" else None
            b_sig_str_acc = int(sig_str_acc[1].replace("%", "")) if sig_str_acc[1] != "---" else None
            # total-str
            total_str = td_1_list[4].text.split() 
            r_total_str_landed = int(total_str[0])
            r_total_str_atmpted = int(total_str[2])
            b_total_str_landed = int(total_str[3])
            b_total_str_atmpted = int(total_str[5])
            # total-str-acc
            r_total_str_acc, b_total_str_acc = None, None
            try:
                r_total_str_acc = int(round(r_total_str_landed / r_total_str_atmpted, 2) * 100)
            except:
                pass
            try:
                b_total_str_acc = int(round(b_total_str_landed / b_total_str_atmpted, 2) * 100)
            except:
                pass
            # TD
            td_players = td_1_list[5].text.split() 
            r_td_landed = int(td_players[0])
            r_td_atmpted = int(td_players[2])
            b_td_landed = int(td_players[3])
            b_td_atmpted = int(td_players[5])
            # td_acc
            td_acc = td_1_list[6].text.split() 
            r_td_acc = int(td_acc[0].replace("%", "")) if td_acc[0] != "---" else None
            b_td_acc = int(td_acc[1].replace("%", "")) if td_acc[1] != "---" else None
            # sub. att
            sub_att = td_1_list[7].text.split()
            r_sub_att, b_sub_att = int(sub_att[0]), int(sub_att[1])
            # rev
            rev = td_1_list[8].text.split()
            r_rev, b_rev = int(rev[0]), int(rev[1])
            # Ctrl
            ctrl = td_1_list[9].text.split()
            r_ctrl = ctrl[0].split(":")
            r_ctrl = int(r_ctrl[0]) * 60 + int(r_ctrl[1]) if r_ctrl[0] != '--' else None
            b_ctrl = ctrl[1].split(":")
            b_ctrl = int(b_ctrl[0]) * 60 + int(b_ctrl[1]) if b_ctrl[0] != '--' else None
            
            # SIG. STR. TABLE
            table2 = tables[1]
            td_2_list = table2.find_all('td', class_ = 'b-fight-details__table-col')
            
            # HEAD
            head_list = td_2_list[3].text.split() 
            r_head_landed = int(head_list[0])
            r_head_atmpted = int(head_list[2])
            b_head_landed = int(head_list[3])
            b_head_atmpted = int(head_list[5])
            # HEAD
            r_head_acc, b_head_acc = None, None
            try:
                r_head_acc = int(round(r_head_landed / r_head_atmpted, 2) * 100)
            except:
                pass
            try:
                b_head_acc = int(round(b_head_landed / b_head_atmpted, 2) * 100)
            except:
                pass
            
            # BODY
            body_list = td_2_list[4].text.split() 
            r_body_landed = int(body_list[0])
            r_body_atmpted = int(body_list[2])
            b_body_landed = int(body_list[3])
            b_body_atmpted = int(body_list[5])
            # BODY ACC
            r_body_acc, b_body_acc = None, None
            try:
                r_body_acc = int(round(r_body_landed / r_body_atmpted, 2) * 100)
            except:
                pass
            try:
                b_body_acc = int(round(b_body_landed / b_body_atmpted, 2) * 100)
            except:
                pass
            
            # LEG
            leg_list = td_2_list[5].text.split() 
            r_leg_landed = int(leg_list[0])
            r_leg_atmpted = int(leg_list[2])
            b_leg_landed = int(leg_list[3])
            b_leg_atmpted = int(leg_list[5])
            # LEG ACC
            r_leg_acc, b_leg_acc = None, None
            try:
                r_leg_acc = int(round(r_leg_landed / r_leg_atmpted, 2) * 100)
            except:
                pass
            try:
                b_leg_acc = int(round(b_leg_landed / b_leg_atmpted, 2) * 100)
            except:
                pass
            
            # DISTANCE
            dist_list = td_2_list[6].text.split() 
            r_dist_landed = int(dist_list[0])
            r_dist_atmpted = int(dist_list[2])
            b_dist_landed = int(dist_list[3])
            b_dist_atmpted = int(dist_list[5])
            # DIST ACC
            r_dist_acc, b_dist_acc = None, None
            try:
                r_dist_acc = int(round(r_dist_landed / r_dist_atmpted, 2) * 100)
            except:
                pass
            try:
                b_dist_acc = int(round(b_dist_landed / b_dist_atmpted, 2) * 100)
            except:
                pass
            
            # CLINCH
            clinch_list = td_2_list[7].text.split() 
            r_clinch_landed = int(clinch_list[0])
            r_clinch_atmpted = int(clinch_list[2])
            b_clinch_landed = int(clinch_list[3])
            b_clinch_atmpted = int(clinch_list[5])
            # CLINCH ACC
            r_clinch_acc, b_clinch_acc = None, None
            try:
                r_clinch_acc = int(round(r_clinch_landed / r_clinch_atmpted, 2) * 100)
            except:
                pass
            try:
                b_clinch_acc = int(round(b_clinch_landed / b_clinch_atmpted, 2) * 100)
            except:
                pass
            
            # Ground
            ground_list = td_2_list[8].text.split() 
            r_ground_landed = int(ground_list[0])
            r_ground_atmpted = int(ground_list[2])
            b_ground_landed = int(ground_list[3])
            b_ground_atmpted = int(ground_list[5])
            # Ground ACC
            r_ground_acc, b_ground_acc = None, None
            try:
                r_ground_acc = int(round(r_ground_landed / r_ground_atmpted, 2) * 100)
            except:
                pass
            try:
                b_ground_acc = int(round(b_ground_landed / b_ground_atmpted, 2) * 100)
            except:
                pass
        else:
            r_kd,b_kd = None, None
            r_sig_str_landed,b_sig_str_landed = None, None
            r_sig_str_atmpted,b_sig_str_atmpted = None, None
            r_sig_str_acc,b_sig_str_acc = None, None
            r_total_str_landed,b_total_str_landed = None, None
            r_total_str_atmpted,b_total_str_atmpted = None, None
            r_total_str_acc,b_total_str_acc = None, None
            r_td_landed,b_td_landed= None, None
            r_td_atmpted,b_td_atmpted = None, None
            r_td_acc,b_td_acc= None, None
            r_sub_att,b_sub_att= None, None
            r_ctrl,b_ctrl= None, None
            
            r_head_landed , b_head_landed = None, None
            r_head_atmpted , b_head_atmpted = None, None
            r_head_acc , b_head_acc = None, None
            r_body_landed , b_body_landed = None, None
            r_body_atmpted , b_body_atmpted = None, None
            r_body_acc , b_body_acc = None, None
            r_leg_landed , b_leg_landed = None, None
            r_leg_atmpted , b_leg_atmpted = None, None
            r_leg_acc , b_leg_acc = None, None
            r_dist_landed , b_dist_landed = None, None
            r_dist_atmpted , b_dist_atmpted = None, None
            r_dist_acc , b_dist_acc = None, None
            r_clinch_landed , b_clinch_landed = None, None
            r_clinch_atmpted , b_clinch_atmpted= None, None
            r_clinch_acc , b_clinch_acc = None, None
            r_ground_landed , b_ground_landed = None, None
            r_ground_atmpted , b_ground_atmpted = None, None
            r_ground_acc , b_ground_acc = None, None
            r_landed_head_per , b_landed_head_per = None, None
            r_landed_body_per , b_landed_body_per= None, None
            r_landed_leg_per , b_landed_leg_per = None, None
            r_landed_dist_per , b_landed_dist_per = None, None
            r_landed_clinch_per , b_landed_clinch_per = None, None
            r_landed_ground_per , b_landed_ground_per = None, None
        
        # LANDED-head&dist
        try:
            r_landed_head_and_dist_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_red b-fight-details__charts-num_pos_left js-red")
            r_landed_head_per = int(r_landed_head_and_dist_list[0].text.strip().replace("%", ""))
            r_landed_dist_per = int(r_landed_head_and_dist_list[1].text.strip().replace("%", ""))
            b_landed_head_and_dist_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_blue b-fight-details__charts-num_pos_right js-blue")
            b_landed_head_per = int(b_landed_head_and_dist_list[0].text.strip().replace("%", ""))
            b_landed_dist_per = int(b_landed_head_and_dist_list[1].text.strip().replace("%", ""))
        except:
            r_landed_head_per, r_landed_dist_per = None, None
            b_landed_head_per, b_landed_dist_per = None, None
        # LANDED-Body&Clinch
        try:
            r_landed_body_and_clinch_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_dark-red b-fight-details__charts-num_pos_left js-red")
            r_landed_body_per = int(r_landed_body_and_clinch_list[0].text.strip().replace("%", ""))
            r_landed_clinch_per = int(r_landed_body_and_clinch_list[1].text.strip().replace("%", ""))
            b_landed_body_and_clinch_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_dark-blue b-fight-details__charts-num_pos_right js-blue")
            b_landed_body_per = int(b_landed_body_and_clinch_list[0].text.strip().replace("%", ""))
            b_landed_clinch_per = int(b_landed_body_and_clinch_list[1].text.strip().replace("%", ""))
        except:
            r_landed_body_per, r_landed_clinch_per = None, None
            b_landed_body_per, b_landed_clinch_per = None, None
            
        # LANDED-leg&ground
        try:
            r_landed_leg_and_ground_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_light-red b-fight-details__charts-num_pos_left js-red")
            r_landed_leg_per = int(r_landed_leg_and_ground_list[0].text.strip().replace("%", ""))
            r_landed_ground_per = int(r_landed_leg_and_ground_list[1].text.strip().replace("%", ""))
            b_landed_leg_and_ground_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_light-blue b-fight-details__charts-num_pos_right js-blue")
            b_landed_leg_per = int(b_landed_leg_and_ground_list[0].text.strip().replace("%", ""))
            b_landed_ground_per = int(b_landed_leg_and_ground_list[1].text.strip().replace("%", ""))
        except:
            # pass
            r_landed_leg_per, r_landed_ground_per = None, None
            b_landed_leg_per, b_landed_ground_per = None, None
            
        # MAKING THE DATA
        data_dic = {
            "event_name" : event_name,
            "event_id" : event_id,
            "fight_id" : fight_id,
            "r_name" : r_name,
            "r_id" : r_id,
            "b_name" : b_name,
            "b_id" : b_id,
            "division" : division_info,
            "title_fight" : is_title_fight,
            "method" : method,
            "finish_round" : finish_round,
            "match_time_sec" : match_time_sec,
            "total_rounds" : total_rounds,
            "referee" : referee,
            "r_kd" : r_kd,
            "r_sig_str_landed" : r_sig_str_landed,
            "r_sig_str_atmpted" : r_sig_str_atmpted,
            "r_sig_str_acc" : r_sig_str_acc,
            "r_total_str_landed" : r_total_str_landed,
            "r_total_str_atmpted" : r_total_str_atmpted,
            "r_total_str_acc" : r_total_str_acc,
            "r_td_landed" : r_td_landed,
            "r_td_atmpted" : r_td_atmpted,
            "r_td_acc" : r_td_acc,
            "r_sub_att" : r_sub_att,
            "r_ctrl" : r_ctrl,
            "r_head_landed" : r_head_landed,
            "r_head_atmpted" : r_head_atmpted,
            "r_head_acc" : r_head_acc,
            "r_body_landed" : r_body_landed,
            "r_body_atmpted" : r_body_atmpted,
            "r_body_acc" : r_body_acc,
            "r_leg_landed" : r_leg_landed,
            "r_leg_atmpted" : r_leg_atmpted,
            "r_leg_acc" : r_leg_acc,
            "r_dist_landed" : r_dist_landed,
            "r_dist_atmpted" : r_dist_atmpted,
            "r_dist_acc" : r_dist_acc,
            "r_clinch_landed" : r_clinch_landed,
            "r_clinch_atmpted" : r_clinch_atmpted,
            "r_clinch_acc" : r_clinch_acc,
            "r_ground_landed" : r_ground_landed,
            "r_ground_atmpted" : r_ground_atmpted,
            "r_ground_acc" : r_ground_acc,
            "r_landed_head_per" : r_landed_head_per,
            "r_landed_body_per" : r_landed_body_per,
            "r_landed_leg_per" : r_landed_leg_per,
            "r_landed_dist_per" : r_landed_dist_per,
            "r_landed_clinch_per" : r_landed_clinch_per,
            "r_landed_ground_per" : r_landed_ground_per,
            "b_kd" : b_kd,
            "b_sig_str_landed" : b_sig_str_landed,
            "b_sig_str_atmpted" : b_sig_str_atmpted,
            "b_sig_str_acc" : b_sig_str_acc,
            "b_total_str_landed" : b_total_str_landed,
            "b_total_str_atmpted" : b_total_str_atmpted,
            "b_total_str_acc" : b_total_str_acc,
            "b_td_landed" : b_td_landed,
            "b_td_atmpted" : b_td_atmpted,
            "b_td_acc" : b_td_acc,
            "b_sub_att" : b_sub_att,
            "b_ctrl" : b_ctrl,
            "b_head_landed" : b_head_landed,
            "b_head_atmpted" : b_head_atmpted,
            "b_head_acc" : b_head_acc,
            "b_body_landed" : b_body_landed,
            "b_body_atmpted" : b_body_atmpted,
            "b_body_acc" : b_body_acc,
            "b_leg_landed" : b_leg_landed,
            "b_leg_atmpted" : b_leg_atmpted,
            "b_leg_acc" : b_leg_acc,
            "b_dist_landed" : b_dist_landed,
            "b_dist_atmpted" : b_dist_atmpted,
            "b_dist_acc" : b_dist_acc,
            "b_clinch_landed" : b_clinch_landed,
            "b_clinch_atmpted" : b_clinch_atmpted,
            "b_clinch_acc" : b_clinch_acc,
            "b_ground_landed" : b_ground_landed,
            "b_ground_atmpted" : b_ground_atmpted,
            "b_ground_acc" : b_ground_acc,
            "b_landed_head_per" : b_landed_head_per,
            "b_landed_body_per" : b_landed_body_per,
            "b_landed_leg_per" : b_landed_leg_per,
            "b_landed_dist_per" : b_landed_dist_per,
            "b_landed_clinch_per" : b_landed_clinch_per,
            "b_landed_ground_per" : b_landed_ground_per
        }
        with lock:
            fight_details.append(data_dic)
            # print(f"Scraped {idx+1}/{len(new_fight_links_all)}: {link}")
            idx += 1
    except requests.exceptions.RequestException as e:
        print(f"Failed {idx}/{new_fight_links_all}: {link} - {str(e)}")
        return

with ThreadPoolExecutor(max_workers= MAX_THREADS) as executor:
    results = [executor.submit(get_fight_data, item) for item in enumerate(new_fight_links_all)]
    for r in results:
        r.result()
        
print(f"Successfully scraped all fight data. Scrapped data {len(fight_details)}")

Successfully scraped all fight data. Scrapped data 8675


In [10]:
df_fight = pd.DataFrame(data=fight_details)
df_fight.to_csv("fight_details.csv", index = False)
df_fight

,event_name,event_id,fight_id,r_name,r_id,b_name,b_id,division,title_fight,method,...,b_clinch_acc,b_ground_landed,b_ground_atmpted,b_ground_acc,b_landed_head_per,b_landed_body_per,b_landed_leg_per,b_landed_dist_per,b_landed_clinch_per,b_landed_ground_per
0,UFC Fight Night: Della Maddalena vs. Prates,872b018076f831b0,5c7c38dece435cde,Beneil Dariush,08af939f41b5a57b,Quillan Salkilld,17734443a833cdf7,lightweight,0,KO/TKO,...,67.0,3.0,3.0,100.0,91.0,8.0,0.0,58.0,16.0,25.0
1,UFC Fight Night: Della Maddalena vs. Prates,872b018076f831b0,eb6d9fef6c374830,Jack Della Maddalena,6b453bc35a823c3f,Carlos Prates,7ee0fd831c0fe7c3,welterweight,0,KO/TKO,...,100.0,14.0,21.0,67.0,75.0,10.0,13.0,85.0,0.0,13.0
2,UFC Fight Night: Della Maddalena vs. Prates,872b018076f831b0,afec383a96893ec5,Tim Elliott,c96d9178c9ed9e62,Steve Erceg,32ab52e5de93092d,flyweight,0,Decision - Unanimous,...,78.0,0.0,0.0,NaN,89.0,10.0,0.0,94.0,5.0,0.0
3,UFC Fight Night: Della Maddalena vs. Prates,872b018076f831b0,cfc884a249141cba,Marwan Rahiki,6eedb757f13b9978,Ollie Schmid,b701f095e6ccb149,featherweight,0,KO/TKO,...,NaN,0.0,0.0,NaN,0.0,33.0,66.0,100.0,0.0,0.0
4,UFC Fight Night: Della Maddalena vs. Prates,872b018076f831b0,a564219e7a964aa6,Shamil Gaziev,6747ccd6d1acd266,Brando Pericic,d0fd0d9ee560dae7,heavyweight,0,KO/TKO,...,62.0,0.0,0.0,NaN,69.0,15.0,14.0,91.0,8.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8670,UFC 3: The American Dream,1a49e0670dfaca31,5167a5aec41f8b6d,Ken Shamrock,63b65af1c5cb02cb,Felix Lee Mitchell,6cbb7661c3258617,open weight,0,Submission,...,100.0,0.0,0.0,NaN,0.0,33.0,66.0,0.0,100.0,0.0
8671,UFC 3: The American Dream,1a49e0670dfaca31,ca837b002a318766,Harold Howard,a2b06ca02bca14c0,Roland Payne,2e04a3b4a2011b97,open weight,0,KO/TKO,...,50.0,0.0,0.0,NaN,0.0,66.0,33.0,66.0,33.0,0.0
8672,UFC 3: The American Dream,1a49e0670dfaca31,f65b000b7e450bcf,Royce Gracie,429e7d3725852ce9,Kimo Leopoldo,08ae5cd9aef7ddd3,open weight,0,Submission,...,100.0,3.0,5.0,60.0,50.0,33.0,16.0,0.0,50.0,50.0
8673,UFC 3: The American Dream,1a49e0670dfaca31,56315d5ac11594a5,Ken Shamrock,63b65af1c5cb02cb,Christophe Leninger,7269329bd87eb479,open weight,0,KO/TKO,...,NaN,0.0,0.0,NaN,100.0,0.0,0.0,100.0,0.0,0.0


# Scrapping the fighter info

In [11]:
r_fighter_id = df_fight['r_id'].unique()
b_fighter_id = df_fight['b_id'].unique()
all_ids = list(set(list(r_fighter_id) + list(b_fighter_id))) # Combining both fighter ids and removing duplicates

base_url = "http://ufcstats.com/fighter-details/" # Base URL for fighter details
def get_fighter_data(item): # Function to scrape fighter data
    """Scrape fighter data from the given link."""
    try:
        idx, id = item
        response = session.get(base_url+id, headers= HEADER, timeout=15)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, "lxml")
        
        # ID FOR MAPPING
        fighter_id = id
        
        # NAMES
        fighter_name = soup.find('span', class_ = 'b-content__title-highlight').text.strip()
        fighter_nick_name = soup.find('p', class_ = "b-content__Nickname").text.strip()
        
        # RECORD DETAILS
        fighter_record = soup.find('span', class_= "b-content__title-record").text.replace("Record:", "").strip().split('-')
        fighter_wins = int(fighter_record[0].split()[0])
        fighter_losses = int(fighter_record[1].split()[0])
        fighter_draws = int(fighter_record[2].split()[0])
        
        # Details
        detail_list = soup.find_all('li', class_ = "b-list__box-list-item b-list__box-list-item_type_block")
        
        try:
            height = detail_list[0].text.replace("Height:", "").strip().replace("'", "").replace('"', '').split()
            height = round((int(height[0]) * 12 + int(height[1])) * 2.54, 2)
            
        except:
            height = None
        
        try:
            weight = detail_list[1].text.replace("Weight:", "").strip().replace(" lbs", "")
            weight = round(float(weight) * 0.45359237, 2)
            
        except:
            weight = None
        
        try:
            reach = detail_list[2].text.replace("Reach:", "").strip().replace('"', "")
            reach = round(int(reach) * 2.54, 2)
        except:
            reach = None
            
        try:
            stance = detail_list[3].text.replace("STANCE:", "").strip()
            stance = stance if stance != "" else None
        except:
            stance = None
        
        try:
            dob = detail_list[4].text.replace("DOB:", "").strip()
            dob = dob if dob != "--" else None
        except:
            dob = None
            
        splm = float(detail_list[5].text.replace("SLpM:", "").strip())
        str_acc = int(detail_list[6].text.replace("Str. Acc.:", "").strip().replace("%", ""))
        sapm = float(detail_list[7].text.replace("SApM:", "").strip())
        str_def = int(detail_list[8].text.replace("Str. Def:", "").strip().replace("%", ""))
        td_avg = float(detail_list[10].text.replace("TD Avg.:", "").strip())
        td_acc = int(detail_list[11].text.replace("TD Acc.:", "").strip().replace("%", ""))
        td_def = int(detail_list[12].text.replace("TD Def.:", "").strip().replace("%", ""))
        sub_avg = float(detail_list[13].text.replace("Sub. Avg.:", "").strip())
        
        # Making The Data
        data_dic = {
            "id" : fighter_id,
            "name" : fighter_name,
            "nick_name" : fighter_nick_name,
            "wins" : fighter_wins,
            "losses" : fighter_losses,
            "draws" : fighter_draws,
            "height" : height,
            "weight" : weight,
            "reach" : reach,
            "stance" : stance,
            "dob" : dob,
            "splm" : splm,
            "str_acc" : str_acc,
            "sapm" : sapm,
            "str_def" : str_def,
            "td_avg" : td_avg,
            "td_avg_acc" : td_acc,
            "td_def" : td_def,
            "sub_avg" : sub_avg
        } 
        with lock:
            fighter_detail_data.append(data_dic)
            # print(f"Scrapped {base_url+id}. {idx+1}/{len(all_ids)}")
            idx += 1
    except:
        print(f"Cannot process the link {base_url+id}. Skipping this link.")
        return
    
with ThreadPoolExecutor(max_workers= MAX_THREADS) as executor:
    results = [executor.submit(get_fighter_data, item) for item in enumerate(all_ids)]
    for r in results:
        r.result()

df_fighter = pd.DataFrame(data= fighter_detail_data)
df_fighter.to_csv("fighter_details.csv", index = False)
print(f"Successfully Scrapped {len(df_fighter)}")

Successfully Scrapped 2689


In [12]:
df_fighter

,id,name,nick_name,wins,losses,draws,height,weight,reach,stance,dob,splm,str_acc,sapm,str_def,td_avg,td_avg_acc,td_def,sub_avg
0,cc98d857b148e33a,Leonardo Guimaraes,Leleco,11,5,0,182.88,83.91,NaN,Orthodox,"Apr 15, 1982",2.35,56,2.99,44,0.00,0,52,1.0
1,7478b7f959ba61f5,Elijah Smith,Swift,10,1,0,175.26,61.23,180.34,Orthodox,"Sep 05, 2002",4.18,48,2.96,46,2.98,53,53,1.1
2,6256b2b1562cb8e1,James Vick,The Texecutioner,13,5,0,190.50,70.31,193.04,Orthodox,"Feb 23, 1987",4.13,39,3.31,60,0.26,33,57,0.7
3,e56daf7725a0b5ab,Justin Edwards,Fast Eddy,9,5,0,177.80,70.31,177.80,Orthodox,"Jan 26, 1983",2.68,42,3.32,51,2.14,26,27,1.0
4,07f959e6596307bb,Carlos Vera,Pequeno,12,4,0,167.64,61.23,175.26,Orthodox,"Nov 05, 1987",1.53,57,2.79,49,0.00,0,25,1.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2684,c2a670b22d2e196f,Aleksa Camur,,6,3,0,185.42,92.99,187.96,Orthodox,"Sep 25, 1995",4.73,57,4.26,54,0.46,22,50,0.0
2685,d34636a85f0f4c90,Motonobu Tezuka,,28,12,7,170.18,61.23,NaN,Southpaw,"Aug 29, 1987",0.87,23,2.80,51,1.50,11,0,0.5
2686,4bdedbdeedff7d1d,Bolaji Oki,The Zulu Warrior,10,4,0,177.80,70.31,185.42,Orthodox,"Nov 15, 1995",6.42,45,5.12,61,1.02,57,75,0.0
2687,1abfb658cd4f8533,Nandor Guelmino,The Hun,11,5,1,190.50,104.33,NaN,NaN,"Dec 20, 1975",3.62,56,3.16,33,0.00,0,42,0.0


# Building the final data, by merging the tables 

In [15]:
df_merger_winners = df_winner.drop(columns=['event_id']).copy() # Copying the winners data to merge later
df_fight = df_fight.merge(right=df_merger_winners, on='fight_id', how='left') # Merging the winners data with fight data

# SAME ROWS HAD DIFF MEANING SO CHANGED THE AVG DATA
df_fighter_renamed__r = df_fighter.add_prefix('r_').drop(columns=['r_name'], errors='ignore') # Renaming the columns for red fighter
df_fighter_renamed__b = df_fighter.add_prefix('b_').drop(columns=['b_name'], errors='ignore') # Renaming the columns for blue fighter

df_fight = df_fight.merge(right=df_fighter_renamed__r, on='r_id', how='left') # Merging the red fighter data
df_fight = df_fight.merge(right=df_fighter_renamed__b, on='b_id', how='left') # Merging the blue fighter data

cols = df_fight.columns

r_cols = [col for col in cols if col.startswith('r_')]
b_cols = [col for col in cols if col.startswith('b_')]  
fighter_cols = r_cols + b_cols

re_ordered_cols = [
    'event_id',
    'event_name',
    'date',
    'location',
    'fight_id',
    'division',
    'title_fight',
    'method',
    'finish_round',
    'match_time_sec',
    'total_rounds',
    'referee'
]

# only keep columns that actually exist (prevents KeyErrors)
re_ordered_cols = [col for col in re_ordered_cols if col in df_fight.columns]

re_ordered_cols += r_cols + b_cols
re_ordered_cols += [col for col in ['winner', 'winner_id'] if col in df_fight.columns]

df_fight = df_fight[re_ordered_cols]

# Converting date and dob to datetime format safely (NO CRASHES)
df_fight['date'] = pd.to_datetime(df_fight['date'], errors='coerce')

if 'r_dob' in df_fight.columns:
    df_fight['r_dob'] = pd.to_datetime(df_fight['r_dob'], errors='coerce')

if 'b_dob' in df_fight.columns:
    df_fight['b_dob'] = pd.to_datetime(df_fight['b_dob'], errors='coerce')

df_fight.to_csv("UFC.csv", index=False)
df_fight

,event_id,event_name,date,location,fight_id,division,title_fight,method,finish_round,match_time_sec,...,b_splm,b_str_acc,b_sapm,b_str_def,b_td_avg,b_td_avg_acc,b_td_def,b_sub_avg,winner,winner_id
0,872b018076f831b0,UFC Fight Night: Della Maddalena vs. Prates,2026-05-02,"Perth, Western Australia, Australia",5c7c38dece435cde,lightweight,0,KO/TKO,1,209,...,5.01,57,3.03,45,7.25,35,80,0.8,Quillan Salkilld,17734443a833cdf7
1,872b018076f831b0,UFC Fight Night: Della Maddalena vs. Prates,2026-05-02,"Perth, Western Australia, Australia",eb6d9fef6c374830,welterweight,0,KO/TKO,3,197,...,4.41,54,4.30,47,0.18,100,77,0.0,Carlos Prates,7ee0fd831c0fe7c3
2,872b018076f831b0,UFC Fight Night: Della Maddalena vs. Prates,2026-05-02,"Perth, Western Australia, Australia",afec383a96893ec5,flyweight,0,Decision - Unanimous,3,300,...,4.64,47,4.07,55,1.13,28,67,0.4,Steve Erceg,32ab52e5de93092d
3,872b018076f831b0,UFC Fight Night: Della Maddalena vs. Prates,2026-05-02,"Perth, Western Australia, Australia",cfc884a249141cba,featherweight,0,KO/TKO,1,167,...,3.23,30,7.90,45,0.00,0,0,0.0,Marwan Rahiki,6eedb757f13b9978
4,872b018076f831b0,UFC Fight Night: Della Maddalena vs. Prates,2026-05-02,"Perth, Western Australia, Australia",a564219e7a964aa6,heavyweight,0,KO/TKO,2,224,...,11.00,57,4.26,58,0.00,0,71,0.0,Brando Pericic,d0fd0d9ee560dae7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8670,1a49e0670dfaca31,UFC 3: The American Dream,1994-09-09,"Charlotte, North Carolina, USA",5167a5aec41f8b6d,open weight,0,Submission,1,274,...,0.00,0,0.00,0,0.00,0,0,0.0,Ken Shamrock,63b65af1c5cb02cb
8671,1a49e0670dfaca31,UFC 3: The American Dream,1994-09-09,"Charlotte, North Carolina, USA",ca837b002a318766,open weight,0,KO/TKO,1,46,...,0.00,0,0.00,0,0.00,0,0,0.0,Harold Howard,a2b06ca02bca14c0
8672,1a49e0670dfaca31,UFC 3: The American Dream,1994-09-09,"Charlotte, North Carolina, USA",f65b000b7e450bcf,open weight,0,Submission,1,280,...,0.76,83,2.12,30,4.55,100,0,2.3,Royce Gracie,429e7d3725852ce9
8673,1a49e0670dfaca31,UFC 3: The American Dream,1994-09-09,"Charlotte, North Carolina, USA",56315d5ac11594a5,open weight,0,KO/TKO,1,289,...,0.00,0,0.00,0,0.00,0,0,0.0,Ken Shamrock,63b65af1c5cb02cb
